In [0]:
alphacollector_credit_overview = dbutils.widgets.get("alphacollector_credit_overview")
date = dbutils.widgets.get("date")
master_aging = dbutils.widgets.get("master_aging")
client = dbutils.widgets.get("client")
office = dbutils.widgets.get("office")
payor = dbutils.widgets.get("payor")
payerdimension = dbutils.widgets.get("payerdimension")
daynumber = dbutils.widgets.get("daynumber")
getdate = dbutils.widgets.get("getdate")

In [0]:
spark.sql(f"TRUNCATE TABLE {alphacollector_credit_overview}")

In [0]:
spark.sql(f"""
WITH thursday AS (
    SELECT 
        try_to_date(CalendarDate, 'MM/dd/yyyy') AS CalendarDate
    FROM {date}
    WHERE DayNumberInWeek = {daynumber}
      AND try_to_date(CalendarDate, 'MM/dd/yyyy') >= DATE {getdate}
)
INSERT INTO {alphacollector_credit_overview}
(
    office_key,
    client_key,
    client_name,
    invoice_number,
    age_from_last_dos,
    bill_date,
    claim_through_date,
    payer_type,
    payer_name,
    payor_key,
    account_balance,
    contact_type,
    account_status,
    collector,
    follow_up_date,
    last_user_note,
    last_user_note_date,
    num_of_touches,
    source_system,
    reporting_date,
    govt_non_govt,
    date_invoice_became_a_credit,
    age_of_credit_balance,
    credit_reporting_balance
)
SELECT
    ofc.OfficeKey,
    clt.ClientKey,
    clt.client_name,
    ma.Invoice_Number,
    ma.Age_from_Last_DOS,
    try_to_date(ma.Bill_Date, 'MM/dd/yyyy') AS Bill_Date,
    try_to_date(ma.Claim_Through_Date, 'MM/dd/yyyy') AS Claim_Through_Date,
    ma.Payer_Type,
    ma.payer_name,
    py.PayorKey,
    ma.account_balance,
    ma.contact_type,
    ma.account_status,
    ma.collector,
    try_to_date(ma.Follow_Up_Date, 'MM/dd/yyyy') AS follow_up_date,
    ma.last_user_note,
    try_to_date(ma.Last_User_Note_Date, 'MM/dd/yyyy') AS last_user_note_date,
    ma.num_of_touches,
    ma.source_system,
    date_add(ma.ReportingDate, -4) AS reporting_date,
    ma.Gov_t___Non_Gov_t,
    try_to_date(ma.Date_Invoice_Became_a_Credit, 'MM/dd/yyyy') AS date_invoice_became_a_credit,
    TRY_CAST(ma.Age_of_Credit_Balance AS INT) AS age_of_credit_balance,
    TRY_CAST(ma.Credit_Reporting_Balance AS DOUBLE) AS credit_reporting_balance
FROM {master_aging} ma
JOIN thursday d
    ON ma.ReportingDate = d.CalendarDate
LEFT JOIN (
    SELECT
        sourcesystemid,
        ClientKey,
        client_name
    FROM (
        SELECT
            sourcesystemid,
            ClientKey,
            CONCAT(conformedfirstname, ' ', conformedlastname) AS client_name,
            ROW_NUMBER() OVER (
                    PARTITION BY sourcesystemid
                    ORDER BY CASE WHEN conformedfirstname IS NOT NULL THEN 0 ELSE 1 END,
                            ClientKey DESC
                ) AS rnb
        FROM {client}
    ) a
    WHERE rnb = 1
) clt
    ON ma.client_number = clt.sourcesystemid
LEFT JOIN {office} ofc
    ON ofc.OfficeNumber = ma.office_number
LEFT JOIN {payor} py
    ON py.payorid = ma.Active_Ins_Code__Bill_To_
WHERE ma.account_balance < 0
  AND ma.source_system = 'Bears'

UNION ALL

SELECT
    ofc.OfficeKey,
    clt.ClientKey,
    clt.client_name,
    ma.Invoice_Number,
    ma.Age_from_Last_DOS,
    try_to_date(ma.Bill_Date, 'MM/dd/yyyy') AS Bill_Date,
    try_to_date(ma.Claim_Through_Date, 'MM/dd/yyyy') AS Claim_Through_Date,
    ma.Payer_Type,
    ma.payer_name,
    py.PayorKey,
    ma.account_balance,
    ma.contact_type,
    ma.account_status,
    ma.collector,
    try_to_date(ma.Follow_Up_Date, 'MM/dd/yyyy') AS follow_up_date,
    ma.last_user_note,
    try_to_date(ma.Last_User_Note_Date, 'MM/dd/yyyy') AS last_user_note_date,
    ma.num_of_touches,
    ma.source_system,
    date_add(ma.ReportingDate, -4) AS reporting_date,
    ma.Gov_t___Non_Gov_t,
    try_to_date(ma.Date_Invoice_Became_a_Credit, 'MM/dd/yyyy') AS date_invoice_became_a_credit,
    TRY_CAST(ma.Age_of_Credit_Balance AS INT) AS age_of_credit_balance,
    TRY_CAST(ma.Credit_Reporting_Balance AS DOUBLE) AS credit_reporting_balance
FROM {master_aging} ma
JOIN thursday d
    ON ma.ReportingDate = d.CalendarDate
LEFT JOIN (
    SELECT
        medicalrecordnumber,
        ClientKey,
        client_name
    FROM (
        SELECT
            medicalrecordnumber,
            ClientKey,
            CONCAT(conformedfirstname, ' ', conformedlastname) AS client_name,
            ROW_NUMBER() OVER (
                PARTITION BY medicalrecordnumber
                ORDER BY medicalrecordnumber
            ) AS rnb
        FROM {client}
    ) a
    WHERE rnb = 1
) clt
    ON ma.client_number = clt.medicalrecordnumber
LEFT JOIN {office} ofc
    ON ofc.OfficeNumber = ma.office_number
LEFT JOIN {payor} py
    ON py.payorid = ma.Active_Ins_Code__Bill_To_
WHERE ma.account_balance < 0
  AND ma.source_system = 'HCHB'

UNION ALL

SELECT
    ofc.OfficeKey,
    clt.ClientKey,
    clt.client_name,
    ma.Invoice_Number,
    ma.Age_from_Last_DOS,
    try_to_date(ma.Bill_Date, 'MM/dd/yyyy') AS Bill_Date,
    try_to_date(ma.Claim_Through_Date, 'MM/dd/yyyy') AS Claim_Through_Date,
    ma.Payer_Type,
    ma.payer_name,
    pd.payerkey,
    ma.account_balance,
    ma.contact_type,
    ma.account_status,
    ma.collector,
    try_to_date(ma.Follow_Up_Date, 'MM/dd/yyyy') AS follow_up_date,
    ma.last_user_note,
    try_to_date(ma.Last_User_Note_Date, 'MM/dd/yyyy') AS last_user_note_date,
    ma.num_of_touches,
    ma.source_system,
    date_add(ma.ReportingDate, -4) AS reporting_date,
    ma.Gov_t___Non_Gov_t,
    try_to_date(ma.Date_Invoice_Became_a_Credit, 'MM/dd/yyyy') AS date_invoice_became_a_credit,
    TRY_CAST(ma.Age_of_Credit_Balance AS INT) AS age_of_credit_balance,
    TRY_CAST(ma.Credit_Reporting_Balance AS DOUBLE) AS credit_reporting_balance
FROM {master_aging} ma
JOIN thursday d
    ON ma.ReportingDate = d.CalendarDate
LEFT JOIN (
    SELECT
        medicalrecordnumber,
        ClientKey,
        client_name
    FROM (
        SELECT
            medicalrecordnumber,
            ClientKey,
            CONCAT(conformedfirstname, ' ', conformedlastname) AS client_name,
            ROW_NUMBER() OVER (
                PARTITION BY medicalrecordnumber
                ORDER BY ClientKey DESC
            ) AS rnb
        FROM {client}
        WHERE SourceSystem = 'CubHub'
    ) a
    WHERE rnb = 1
) clt
    ON ma.client_number = clt.medicalrecordnumber
LEFT JOIN {office} ofc
    ON ofc.OfficeNumber = ma.office_number
LEFT JOIN (
    SELECT
        payerkey,
        name
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY name
                ORDER BY payerkey DESC
            ) AS rnb
        FROM {payerdimension}
        WHERE sourcesystemkey = 19
    ) p
    WHERE rnb = 1
) pd
    ON pd.name = ma.payer_name
WHERE ma.account_balance < 0
  AND ma.source_system = 'CubHub'
""")